<a href="https://colab.research.google.com/github/GopalKrishna-India/Geospatial/blob/master/Renaming%26FilteringFieldData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**A clean, robust Python script using pandas that reads your field data, filters it against your list of 20 target botanical names, and accounts for potential spelling typos in local names using fuzzy string matching with Summary. Keeping any record of Eucalytus as 'Eucalytus spp'**

In [9]:
import pandas as pd
from collections import Counter
from difflib import get_close_matches

# -------------------------------------------------------------------
# 1. Configuration & Input Data Lists
# -------------------------------------------------------------------
input_file = "NicheMod_Combined_CLEANED_V2.xlsx"        # Input file (.csv or .xlsx)
output_file = "NicheMod_Combined_CLEANED_V2filtered_data.csv"    # Output filtered file path
summary_file = "summary_report.txt"  # File path to save text summary

In [11]:
# Target 20 botanical species
target_botanical_names = [
    "Acacia nilotica",
    "Ailanthus excelsa",
    "Pongamia pinnata",
    "Populus deltoides",
    "Prosopis cineraria",
    "Terminalia arjuna",
    "Acacia mangium",
    "Gliricidia sepium",
    "Anthocephalus cadamba",
    "Grewia optiva",
    "Mangifera indica",
    "Azadirachta indica",
    "Albizia lebbeck",
    "Casuarina equisetifolia",
    "Dalbergia sissoo",
    "Bambusa vulgaris",
    "Eucalyptus spp.",
    "Gmelina arborea",
    "Melia dubia",
    "Tectona grandis"
]

# Raw local name mappings with synonyms
raw_local_mapping = {
    "Babul / Babool / Kikar": "Acacia nilotica",
    "Subabul / Soobabul / Subavul": "Leucaena leucocephala",
    "Maharukh / Mahanimb / Araduso": "Ailanthus excelsa",
    "Pungam / Karanja / Indian beech": "Pongamia pinnata",
    "Poplar": "Populus deltoides",
    "Khejri / Sangri": "Prosopis cineraria",
    "Arjun / Arjuna": "Terminalia arjuna",
    "Mangium / Black wattle": "Acacia mangium",
    "Seema konna / Mexican lilac": "Gliricidia sepium",
    "Kadamb / Kadamba": "Anthocephalus cadamba",
    "Bhimal / Bheemal": "Grewia optiva",
    "Mango / Aam": "Mangifera indica",
    "Neem": "Azadirachta indica",
    "Siris / Lebbek": "Albizia lebbeck",
    "Casuarina / Jangli Saru": "Casuarina equisetifolia",
    "Indian Rosewood / Shisham / Sissoo": "Dalbergia sissoo",
    "Bamboo / Bans": "Bambusa vulgaris",
    "Safeda / Eucalyptus / Nilgiri": "Eucalyptus spp.",
    "Gmelina / Gamhar / Khamara": "Gmelina arborea",
    "Malabar Neem": "Melia dubia",
    "Teak / Sagwan": "Tectona grandis"
}

# Expand mapping for individual synonyms
local_name_mapping = {}
for synonyms, botanical in raw_local_mapping.items():
    for name in synonyms.split("/"):
        clean_name = name.strip()
        if clean_name:
            local_name_mapping[clean_name] = botanical


# -------------------------------------------------------------------
# 2. Fuzzy Matching Helper Function
# -------------------------------------------------------------------
def match_local_name(local_val, mapping_dict, cutoff=0.65):
    """Matches local names (handling typos) to target botanical names."""
    if pd.isna(local_val) or not str(local_val).strip():
        return None

    clean_val = str(local_val).strip()

    # Exact case-insensitive lookup
    for key, botanical in mapping_dict.items():
        if clean_val.lower() == key.lower():
            return botanical

    # Fuzzy match for typos
    known_keys = list(mapping_dict.keys())
    matches = get_close_matches(clean_val, known_keys, n=1, cutoff=cutoff)
    if matches:
        return mapping_dict[matches[0]]

    return None


# -------------------------------------------------------------------
# 3. Processing Core Logic
# -------------------------------------------------------------------
def process_field_data():
    if input_file.endswith(".csv"):
        df = pd.read_csv(input_file)
    else:
        df = pd.read_excel(input_file)

    total_raw_records = len(df)

    botanical_col = "Botanical_Name"
    local_col = "Local_Name"

    canonical_botanical = {name.lower(): name for name in target_botanical_names}

    direct_botanical_matches = 0
    inferred_from_local_matches = 0
    eucalyptus_fallback_matches = 0

    final_botanical_list = []

    for idx, row in df.iterrows():
        raw_botanical = str(row.get(botanical_col, '')).strip() if pd.notna(row.get(botanical_col)) else ""
        raw_local = str(row.get(local_col, '')).strip() if pd.notna(row.get(local_col)) else ""

        matched_species = None

        # STEP 1: Check Local Name FIRST to resolve conflicts correctly (e.g. Malabar Neem -> Melia dubia)
        if raw_local:
            inferred = match_local_name(raw_local, local_name_mapping)
            if inferred:
                matched_species = inferred
                inferred_from_local_matches += 1

        # STEP 2: Direct Match on Exact Target Botanical Name
        if not matched_species and raw_botanical.lower() in canonical_botanical:
            matched_species = canonical_botanical[raw_botanical.lower()]
            direct_botanical_matches += 1

        # STEP 3: Fallback check for any wildcard Eucalyptus variant (if local name didn't match anything else)
        if not matched_species:
            if "eucalyptus" in raw_botanical.lower() or "eucalyptus" in raw_local.lower():
                matched_species = "Eucalyptus spp."
                eucalyptus_fallback_matches += 1

        final_botanical_list.append(matched_species)

    # Assign resolved species
    df['Resolved_Botanical_Name'] = final_botanical_list

    # Keep only matched records
    filtered_df = df[df['Resolved_Botanical_Name'].notna()].copy()
    filtered_df[botanical_col] = filtered_df['Resolved_Botanical_Name']
    filtered_df.drop(columns=['Resolved_Botanical_Name'], inplace=True)

    # Summary Metrics
    total_matched_records = len(filtered_df)
    total_discarded_records = total_raw_records - total_matched_records
    retention_rate = (total_matched_records / total_raw_records * 100) if total_raw_records > 0 else 0
    species_counts = Counter(filtered_df[botanical_col])

    # Save Output
    if output_file.endswith(".xlsx"):
        filtered_df.to_excel(output_file, index=False)
    else:
        filtered_df.to_csv(output_file, index=False)

    # Printable Report
    summary_lines = []
    summary_lines.append("=" * 60)
    summary_lines.append("               FIELD DATA FILTERING SUMMARY               ")
    summary_lines.append("=" * 60)
    summary_lines.append(f"Total Raw Input Records     : {total_raw_records:,}")
    summary_lines.append(f"Total Useful/Matched Records : {total_matched_records:,} ({retention_rate:.2f}%)")
    summary_lines.append(f"Total Discarded/Rejected    : {total_discarded_records:,} ({100 - retention_rate:.2f}%)")
    summary_lines.append("-" * 60)
    summary_lines.append("MATCH BREAKDOWN:")
    summary_lines.append(f"  • Matched from Local Name (Preferred) : {inferred_from_local_matches:,}")
    summary_lines.append(f"  • Direct Botanical Name Match         : {direct_botanical_matches:,}")
    summary_lines.append(f"  • Eucalyptus Wildcard Fallback        : {eucalyptus_fallback_matches:,}")
    summary_lines.append("-" * 60)
    summary_lines.append("SPECIES-WISE USEFUL RECORD COUNTS:")
    summary_lines.append(f"{'Botanical Name':<32} | {'Count':<8} | {'Share (%)'}")
    summary_lines.append("-" * 60)

    for species in target_botanical_names:
        count = species_counts.get(species, 0)
        pct = (count / total_matched_records * 100) if total_matched_records > 0 else 0
        summary_lines.append(f"{species:<32} | {count:<8} | {pct:.2f}%")

    summary_lines.append("=" * 60)

    summary_text = "\n".join(summary_lines)
    print(summary_text)

    with open(summary_file, "w", encoding="utf-8") as f:
        f.write(summary_text)


if __name__ == "__main__":
    process_field_data()

               FIELD DATA FILTERING SUMMARY               
Total Raw Input Records     : 107,834
Total Useful/Matched Records : 105,850 (98.16%)
Total Discarded/Rejected    : 1,984 (1.84%)
------------------------------------------------------------
MATCH BREAKDOWN:
  • Matched from Local Name (Preferred) : 105,847
  • Direct Botanical Name Match         : 3
  • Eucalyptus Wildcard Fallback        : 0
------------------------------------------------------------
SPECIES-WISE USEFUL RECORD COUNTS:
Botanical Name                   | Count    | Share (%)
------------------------------------------------------------
Acacia nilotica                  | 438      | 0.41%
Ailanthus excelsa                | 41       | 0.04%
Pongamia pinnata                 | 45       | 0.04%
Populus deltoides                | 2        | 0.00%
Prosopis cineraria               | 24       | 0.02%
Terminalia arjuna                | 29       | 0.03%
Acacia mangium                   | 0        | 0.00%
Gliricidia sepium 